# Tournament Manager — Repository Tests

This notebook tests every repository method using an **in-memory SQLite** database.
No external services are required.

Run all cells in order from top to bottom.

## 0. Setup — create in-memory DB and tables

In [ ]:
import sys, os

# Make project root importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Force test environment (SQLite in-memory)
os.environ["APP_ENV"] = "test"
os.environ["DATABASE_URL_TEST"] = "sqlite:///:memory:"

from sqlalchemy import create_engine, event
from sqlalchemy.orm import sessionmaker

from src.domain.models import Base
from src.domain.repositories import UnitOfWork

# Create engine + tables
engine = create_engine(
    "sqlite:///:memory:",
    connect_args={"check_same_thread": False},
    echo=False,
)

# Enable FK support in SQLite
@event.listens_for(engine, "connect")
def set_sqlite_pragma(dbapi_conn, connection_record):
    cursor = dbapi_conn.cursor()
    cursor.execute("PRAGMA foreign_keys=ON")
    cursor.close()

Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine, autoflush=False, autocommit=False)

def make_uow():
    """Return a UnitOfWork wired to the test engine."""
    return UnitOfWork(session_factory=Session)

print("✅ Database and tables created successfully")

---
## 1. FighterRepository

### 1.1 Create (add)

In [ ]:
from src.domain.models import Fighter

with make_uow() as uow:
    f1 = Fighter(
        first_name="Ana",
        last_name="García",
        gender="F",
        belt_level="blue",
        club_name="Club Judo BCN",
        country="Spain",
        email="ana.garcia@example.com",
    )
    f2 = Fighter(
        first_name="Carlos",
        last_name="López",
        gender="M",
        belt_level="black",
        club_name="Dojo Madrid",
        country="Spain",
        email="carlos.lopez@example.com",
    )
    f3 = Fighter(
        first_name="Yuki",
        last_name="Tanaka",
        gender="F",
        belt_level="purple",
        club_name="Osaka Dojo",
        country="Japan",
        email="yuki.tanaka@example.com",
    )
    uow.fighters.add(f1)
    uow.fighters.add(f2)
    uow.fighters.add(f3)
    uow.commit()
    print(f"✅ Created fighters: {f1}, {f2}, {f3}")
    fighter_ids = [f1.fighter_id, f2.fighter_id, f3.fighter_id]

print("Fighter IDs:", fighter_ids)

### 1.2 Read by ID (get)

In [ ]:
with make_uow() as uow:
    fighter = uow.fighters.get(fighter_ids[0])
    assert fighter is not None, "Fighter should exist"
    assert fighter.first_name == "Ana"
    print(f"✅ get() → {fighter}")

### 1.3 Read all (get_all)

In [ ]:
with make_uow() as uow:
    all_fighters = uow.fighters.get_all()
    assert len(all_fighters) == 3
    print(f"✅ get_all() → {len(all_fighters)} fighters: {[f.last_name for f in all_fighters]}")

### 1.4 Update

In [ ]:
with make_uow() as uow:
    fighter = uow.fighters.get(fighter_ids[0])
    fighter.belt_level = "purple"
    uow.fighters.update(fighter)
    uow.commit()

with make_uow() as uow:
    updated = uow.fighters.get(fighter_ids[0])
    assert updated.belt_level == "purple"
    print(f"✅ update() → belt_level now '{updated.belt_level}'")

### 1.5 Specific queries — get_by_last_name, get_by_belt_level, get_by_country

In [ ]:
with make_uow() as uow:
    results = uow.fighters.get_by_last_name("García")
    assert len(results) == 1
    print(f"✅ get_by_last_name('García') → {results[0]}")

    results = uow.fighters.get_by_belt_level("black")
    assert len(results) == 1
    print(f"✅ get_by_belt_level('black') → {results[0]}")

    results = uow.fighters.get_by_country("Spain")
    assert len(results) == 2
    print(f"✅ get_by_country('Spain') → {len(results)} fighters")

### 1.6 Pagination — get_paginated

In [ ]:
with make_uow() as uow:
    page1 = uow.fighters.get_paginated(page=1, page_size=2)
    page2 = uow.fighters.get_paginated(page=2, page_size=2)
    total  = uow.fighters.count()

    assert len(page1) == 2, f"Expected 2 on page 1, got {len(page1)}"
    assert len(page2) == 1, f"Expected 1 on page 2, got {len(page2)}"
    assert total == 3
    print(f"✅ get_paginated() page1={[f.last_name for f in page1]}, page2={[f.last_name for f in page2]}")
    print(f"✅ count() → {total}")

### 1.7 Domain operation — add_medical_record

In [ ]:
with make_uow() as uow:
    record = uow.fighters.add_medical_record(
        fighter_id=fighter_ids[0],
        blood_type="A+",
        notes="No known conditions",
        doctor_name="Dr. Martínez",
    )
    uow.commit()
    print(f"✅ add_medical_record() → {record}")

# Verify record is attached
with make_uow() as uow:
    fighter = uow.fighters.get(fighter_ids[0])
    assert fighter.medical_record is not None
    assert fighter.medical_record.blood_type == "A+"
    print(f"✅ Medical record attached: blood_type={fighter.medical_record.blood_type}")

### 1.8 Delete

In [ ]:
with make_uow() as uow:
    fighter = uow.fighters.get(fighter_ids[2])   # Yuki Tanaka
    uow.fighters.delete(fighter)
    uow.commit()

with make_uow() as uow:
    assert uow.fighters.get(fighter_ids[2]) is None
    assert uow.fighters.count() == 2
    print("✅ delete() → fighter removed successfully")

---
## 2. TournamentRepository

### 2.1 Create

In [ ]:
from datetime import datetime, timedelta
from src.domain.models import Tournament

now = datetime.utcnow()

with make_uow() as uow:
    t1 = Tournament(
        name="Copa Primavera 2026",
        location="Barcelona",
        start_date=now + timedelta(days=30),
        end_date=now + timedelta(days=31),
        max_fighters=64,
    )
    t2 = Tournament(
        name="Grand Prix Madrid",
        location="Madrid",
        start_date=now + timedelta(days=60),
        end_date=now + timedelta(days=61),
        max_fighters=128,
    )
    uow.tournaments.add(t1)
    uow.tournaments.add(t2)
    uow.commit()
    tournament_ids = [t1.tournament_id, t2.tournament_id]
    print(f"✅ Created tournaments: {t1}, {t2}")

### 2.2 Read by ID, Read all

In [ ]:
with make_uow() as uow:
    t = uow.tournaments.get(tournament_ids[0])
    assert t.name == "Copa Primavera 2026"
    print(f"✅ get() → {t}")

    all_t = uow.tournaments.get_all()
    assert len(all_t) == 2
    print(f"✅ get_all() → {len(all_t)} tournaments")

### 2.3 Update

In [ ]:
with make_uow() as uow:
    t = uow.tournaments.get(tournament_ids[0])
    t.max_fighters = 80
    uow.tournaments.update(t)
    uow.commit()

with make_uow() as uow:
    t = uow.tournaments.get(tournament_ids[0])
    assert t.max_fighters == 80
    print(f"✅ update() → max_fighters now {t.max_fighters}")

### 2.4 Specific queries — get_by_name, get_by_location, get_active_tournaments

In [ ]:
with make_uow() as uow:
    results = uow.tournaments.get_by_name("Copa")
    assert len(results) == 1
    print(f"✅ get_by_name('Copa') → {results[0].name}")

    results = uow.tournaments.get_by_location("Madrid")
    assert len(results) == 1
    print(f"✅ get_by_location('Madrid') → {results[0].name}")

    active = uow.tournaments.get_active_tournaments()
    assert len(active) >= 2
    print(f"✅ get_active_tournaments() → {len(active)} active")

### 2.5 Domain operation — add_tatami_to_tournament

In [ ]:
with make_uow() as uow:
    tatami = uow.tournaments.add_tatami_to_tournament(
        tournament_id=tournament_ids[0],
        name="Tatami A",
        mat_number=1,
        area_size=64.0,
    )
    uow.commit()
    print(f"✅ add_tatami_to_tournament() → {tatami}")

### 2.6 Delete

In [ ]:
with make_uow() as uow:
    t = uow.tournaments.get(tournament_ids[1])
    uow.tournaments.delete(t)
    uow.commit()

with make_uow() as uow:
    assert uow.tournaments.get(tournament_ids[1]) is None
    print("✅ delete() → tournament removed")

---
## 3. CategoryRepository

### 3.1 Create

In [ ]:
from src.domain.models import Category

with make_uow() as uow:
    c1 = Category(name="Blue -57kg Male", belt_level="blue", weight_min_kg=0, weight_max_kg=57, gender="M")
    c2 = Category(name="Black -70kg Female", belt_level="black", weight_min_kg=57, weight_max_kg=70, gender="F")
    c3 = Category(name="Blue -70kg Female", belt_level="blue", weight_min_kg=57, weight_max_kg=70, gender="F")
    uow.categories.add(c1)
    uow.categories.add(c2)
    uow.categories.add(c3)
    uow.commit()
    cat_ids = [c1.category_id, c2.category_id, c3.category_id]
    print(f"✅ Created categories: {c1}, {c2}, {c3}")

### 3.2 Read by ID and Read all

In [ ]:
with make_uow() as uow:
    c = uow.categories.get(cat_ids[0])
    assert c.belt_level == "blue"
    print(f"✅ get() → {c}")

    all_c = uow.categories.get_all()
    assert len(all_c) == 3
    print(f"✅ get_all() → {len(all_c)} categories")

### 3.3 Update

In [ ]:
with make_uow() as uow:
    c = uow.categories.get(cat_ids[0])
    c.weight_max_kg = 60
    uow.categories.update(c)
    uow.commit()

with make_uow() as uow:
    c = uow.categories.get(cat_ids[0])
    assert c.weight_max_kg == 60
    print(f"✅ update() → weight_max_kg now {c.weight_max_kg}")

### 3.4 Specific queries — get_by_belt_level, get_by_gender, get_by_weight_range

In [ ]:
with make_uow() as uow:
    results = uow.categories.get_by_belt_level("blue")
    assert len(results) == 2
    print(f"✅ get_by_belt_level('blue') → {[c.name for c in results]}")

    results = uow.categories.get_by_gender("F")
    assert len(results) == 2
    print(f"✅ get_by_gender('F') → {[c.name for c in results]}")

    results = uow.categories.get_by_weight_range(max_kg=70)
    assert len(results) >= 2
    print(f"✅ get_by_weight_range(max_kg=70) → {[c.name for c in results]}")

### 3.5 Delete

In [ ]:
with make_uow() as uow:
    c = uow.categories.get(cat_ids[2])
    uow.categories.delete(c)
    uow.commit()

with make_uow() as uow:
    assert uow.categories.get(cat_ids[2]) is None
    print("✅ delete() → category removed")

---
## 4. RegistrationRepository

### 4.1 Create registrations

In [ ]:
from src.domain.models import FighterCategoryRegistration

# Refresh IDs (fighter_ids[2] was deleted, use 0 and 1)
fid1 = fighter_ids[0]
fid2 = fighter_ids[1]
tid  = tournament_ids[0]
cid1 = cat_ids[0]
cid2 = cat_ids[1]

with make_uow() as uow:
    reg1 = FighterCategoryRegistration(
        fighter_id=fid1, category_id=cid1, tournament_id=tid,
        weight_in_weight_kg=56.5, is_approved=False,
    )
    reg2 = FighterCategoryRegistration(
        fighter_id=fid2, category_id=cid2, tournament_id=tid,
        weight_in_weight_kg=68.0, is_approved=False,
    )
    uow.registrations.add(reg1)
    uow.registrations.add(reg2)
    uow.commit()
    print(f"✅ Created registrations: {reg1}, {reg2}")

### 4.2 Read by composite PK

In [ ]:
with make_uow() as uow:
    reg = uow.registrations.get((fid1, cid1, tid))
    assert reg is not None
    assert reg.weight_in_weight_kg == 56.5
    print(f"✅ get() by composite PK → {reg}")

### 4.3 Read all

In [ ]:
with make_uow() as uow:
    all_regs = uow.registrations.get_all()
    assert len(all_regs) == 2
    print(f"✅ get_all() → {len(all_regs)} registrations")

### 4.4 Update — change weight

In [ ]:
with make_uow() as uow:
    reg = uow.registrations.get((fid1, cid1, tid))
    reg.weight_in_weight_kg = 55.8
    uow.registrations.update(reg)
    uow.commit()

with make_uow() as uow:
    reg = uow.registrations.get((fid1, cid1, tid))
    assert reg.weight_in_weight_kg == 55.8
    print(f"✅ update() → weight_in_weight_kg now {reg.weight_in_weight_kg}")

### 4.5 Domain operation — approve_registration

In [ ]:
with make_uow() as uow:
    reg = uow.fighters.approve_registration(
        fighter_id=fid1, category_id=cid1, tournament_id=tid
    )
    uow.commit()
    print(f"✅ approve_registration() → is_approved={reg.is_approved}")

with make_uow() as uow:
    pending = uow.registrations.get_pending_approvals(tid)
    assert len(pending) == 1   # reg2 still pending
    print(f"✅ get_pending_approvals() → {len(pending)} pending")

### 4.6 Specific queries

In [ ]:
with make_uow() as uow:
    by_tournament = uow.registrations.get_by_tournament(tid)
    assert len(by_tournament) == 2
    print(f"✅ get_by_tournament() → {len(by_tournament)} registrations")

    by_fighter = uow.registrations.get_by_fighter(fid1)
    assert len(by_fighter) == 1
    print(f"✅ get_by_fighter() → {len(by_fighter)} registrations")

### 4.7 Delete

In [ ]:
with make_uow() as uow:
    reg = uow.registrations.get((fid2, cid2, tid))
    uow.registrations.delete(reg)
    uow.commit()

with make_uow() as uow:
    assert uow.registrations.get((fid2, cid2, tid)) is None
    print("✅ delete() → registration removed")

---
## 5. MatchRepository

### 5.1 Create matches

In [ ]:
from datetime import datetime, timedelta
from src.domain.models import Match

now = datetime.utcnow()

with make_uow() as uow:
    m1 = Match(
        tournament_id=tournament_ids[0],
        category_id=cat_ids[0],
        blue_fighter_id=fighter_ids[0],
        red_fighter_id=fighter_ids[1],
        winner_id=fighter_ids[0],
        round_number=1,
        duration_seconds=180.0,
        win_method="ippon",
        scheduled_time=now + timedelta(hours=2),
    )
    m2 = Match(
        tournament_id=tournament_ids[0],
        category_id=cat_ids[0],
        blue_fighter_id=fighter_ids[0],
        red_fighter_id=fighter_ids[1],
        winner_id=fighter_ids[1],
        round_number=2,
        duration_seconds=240.0,
        win_method="waza-ari",
        scheduled_time=now + timedelta(hours=4),
    )
    uow.matches.add(m1)
    uow.matches.add(m2)
    uow.commit()
    match_ids = [m1.match_id, m2.match_id]
    print(f"✅ Created matches: {m1}, {m2}")

### 5.2 Read by ID and Read all

In [ ]:
with make_uow() as uow:
    m = uow.matches.get(match_ids[0])
    assert m.win_method == "ippon"
    print(f"✅ get() → {m}")

    all_m = uow.matches.get_all()
    assert len(all_m) == 2
    print(f"✅ get_all() → {len(all_m)} matches")

### 5.3 Update

In [ ]:
with make_uow() as uow:
    m = uow.matches.get(match_ids[0])
    m.duration_seconds = 195.0
    uow.matches.update(m)
    uow.commit()

with make_uow() as uow:
    m = uow.matches.get(match_ids[0])
    assert m.duration_seconds == 195.0
    print(f"✅ update() → duration_seconds now {m.duration_seconds}")

### 5.4 Specific queries

In [ ]:
with make_uow() as uow:
    by_tournament = uow.matches.get_by_tournament(tournament_ids[0])
    assert len(by_tournament) == 2
    print(f"✅ get_by_tournament() → {len(by_tournament)} matches")

    by_fighter = uow.matches.get_by_fighter(fighter_ids[0])
    assert len(by_fighter) == 2   # Ana was in both
    print(f"✅ get_by_fighter() → {len(by_fighter)} matches")

    by_category = uow.matches.get_by_category(cat_ids[0])
    assert len(by_category) == 2
    print(f"✅ get_by_category() → {len(by_category)} matches")

    round1 = uow.matches.get_by_round(tournament_ids[0], 1)
    assert len(round1) == 1
    print(f"✅ get_by_round(round=1) → {len(round1)} match")

    wins = uow.matches.get_wins_by_fighter(fighter_ids[0])
    assert len(wins) == 1
    print(f"✅ get_wins_by_fighter() → {len(wins)} win for fighter {fighter_ids[0]}")

### 5.5 Delete

In [ ]:
with make_uow() as uow:
    m = uow.matches.get(match_ids[1])
    uow.matches.delete(m)
    uow.commit()

with make_uow() as uow:
    assert uow.matches.get(match_ids[1]) is None
    print("✅ delete() → match removed")

---
## 6. Rollback test

In [ ]:
from src.domain.models import Fighter

count_before = None
with make_uow() as uow:
    count_before = uow.fighters.count()

try:
    with make_uow() as uow:
        uow.fighters.add(Fighter(first_name="Ghost", last_name="Fighter"))
        uow.fighters.add(Fighter(first_name="Ghost2", last_name="Fighter2"))
        # Deliberately raise an exception to trigger rollback
        raise RuntimeError("Simulated error — should rollback")
except RuntimeError:
    pass

with make_uow() as uow:
    count_after = uow.fighters.count()

assert count_after == count_before, (
    f"Rollback failed! Before={count_before}, After={count_after}"
)
print(f"✅ Rollback works — count unchanged at {count_after}")

---
## ✅ All tests passed!

In [ ]:
print("=" * 50)
print("ALL REPOSITORY TESTS PASSED SUCCESSFULLY ✅")
print("=" * 50)